# Ejemplo: Usando la Conexión Centralizada

Este notebook demuestra cómo usar el módulo `conexion_sqlserver` en tus propios notebooks.

## Paso 1: Importar el módulo de conexión

In [ ]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO
from conexion_sqlserver import ANIO_INICIO, ANIO_FIN

# Obtener los engines
engine_origen, engine_destino = obtener_engines()

print(f'\n=== CONFIGURACION ==="')
print(f'Servidor: {SERVIDOR}')
print(f'Usuario: {USUARIO}')
print(f'Rango temporal: {ANIO_INICIO} - {ANIO_FIN}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')

## Paso 2: Usar los engines para lectura

In [ ]:
import pandas as pd
from sqlalchemy import text

# Leer datos del servidor de origen
consulta = """
SELECT TOP 5 
    CustomerID, CustomerName, CityID, DeliveryPostalCode
FROM Sales.Customers
ORDER BY CustomerID
"""

df = pd.read_sql_query(consulta, engine_origen)
print(f'Se leyeron {len(df)} clientes:')
display(df)

## Paso 3: Crear un esquema en el destino

In [ ]:
# Crear esquema 'dw' si no existe
sql_crear_esquema = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'dw')
    EXEC('CREATE SCHEMA dw');
"""

with engine_destino.begin() as conn:
    conn.execute(text(sql_crear_esquema))
    print('Esquema dw creado o ya existente')

## Paso 4: Insertar datos en el destino

In [ ]:
# Crear una tabla de ejemplo en el destino
sql_crear_tabla = """
IF OBJECT_ID('dw.clientes_ejemplo', 'U') IS NULL
BEGIN
    CREATE TABLE dw.clientes_ejemplo (
        customer_id INT PRIMARY KEY,
        customer_name NVARCHAR(100),
        city_id INT,
        postal_code NVARCHAR(20),
        fecha_carga DATETIME DEFAULT GETDATE()
    );
END;
"""

with engine_destino.begin() as conn:
    conn.execute(text(sql_crear_tabla))
    print('Tabla dw.clientes_ejemplo creada o ya existente')

## Paso 5: Insertar los datos leídos

In [ ]:
# Renombrar columnas para que coincidan con la tabla destino
df_cargable = df.rename(columns={
    'CustomerID': 'customer_id',
    'CustomerName': 'customer_name',
    'CityID': 'city_id',
    'DeliveryPostalCode': 'postal_code'
})

# Insertar datos (reemplazar si existen)
try:
    df_cargable.to_sql(
        'clientes_ejemplo',
        engine_destino,
        schema='dw',
        if_exists='append',  # o 'replace' para sobrescribir
        index=False
    )
    print(f'Se insertaron {len(df_cargable)} registros en dw.clientes_ejemplo')
except Exception as e:
    print(f'Error al insertar: {e}')

## Paso 6: Verificar los datos cargados

In [ ]:
# Leer desde la tabla destino para verificar
consulta_verificacion = """
SELECT customer_id, customer_name, postal_code, fecha_carga
FROM dw.clientes_ejemplo
ORDER BY customer_id
"""

df_verificado = pd.read_sql_query(consulta_verificacion, engine_destino)
print(f'Registros en dw.clientes_ejemplo: {len(df_verificado)}')
display(df_verificado)

## Paso 7: Ejecutar sentencias SQL complejas

In [ ]:
# Ejemplo: generar estadísticas
sql_estadisticas = """
SELECT 
    COUNT(*) AS total_registros,
    COUNT(DISTINCT customer_id) AS clientes_unicos,
    COUNT(DISTINCT city_id) AS ciudades
FROM dw.clientes_ejemplo
"""

with engine_destino.connect() as conn:
    resultado = conn.execute(text(sql_estadisticas)).mappings().first()
    print('\n=== ESTADISTICAS ===')
    for clave, valor in resultado.items():
        print(f'{clave}: {valor}')

## Notas Importantes

1. **Los engines son reutilizables**: Una vez obtenidos, puedes usarlos múltiples veces en el notebook
2. **Contextos transaccionales**: Usa `with engine.begin()` para transacciones o `with engine.connect()` para lectura
3. **Manejo de errores**: Siempre rodea operaciones de inserción con try/except
4. **Desempeño**: Para grandes volúmenes, considera usar `if_exists='append'` y batch inserts
5. **Auditoría**: La tabla destino incluye `fecha_carga` automáticamente para rastrear cuándo se insertó cada registro